# 🔬 POT: Prototypical Optimal Transport para Segmentación Semántica Débilmente Supervisada
## Laboratorio 2 — Implementación sobre Cityscapes

**Paper de referencia:** *POT: Prototypical Optimal Transport for Weakly Supervised Semantic Segmentation* (CVPR 2025)

---
### Pipeline general:
1. **Feature Partitioning** → Generar CAMs y agrupar features en K clusters por clase (k-means)
2. **Feature Allocation** → Asignar features a clusters usando Optimal Transport con restricción similarity-aware (Sinkhorn)
3. **Feature Activation** → Regenear prototipos, activar por cosine similarity, reponderar con plan de transporte
4. **Loss total:** `L_cls` (multi-label soft margin) + `ϑ * L_otcon` (OT consistency loss)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 📦 SECCIÓN 0 — Instalación de dependencias

Instalamos las librerías necesarias. `POT` (Python Optimal Transport) es la librería principal para resolver
el problema de Sinkhorn OT de forma eficiente. `pydensecrf` se usa para post-procesar las máscaras con DenseCRF.

In [ ]:
# ── Instalación de librerías (Sección 0 — versión final estable) ─────────────
!pip install scikit-learn -q
!pip install matplotlib seaborn -q
!pip install torchmetrics -q

print('✅ Dependencias instaladas correctamente')

✅ Dependencias instaladas correctamente


---
## ⚙️ SECCIÓN 1 — Imports y configuración global

Aquí definimos todos los hiperparámetros del paper:
- `K=3` clusters por clase (Sección 3.1 del paper)
- `CAM_THRESHOLD=0.7` para seleccionar features confiables
- `THETA=3` peso de la consistency loss (Eq. 15)
- `LR_ENCODER=0.1`, `LR_EXTRA=1` (tasas de aprendizaje del paper)
- `NUM_CLASSES=19` clases de Cityscapes (sin background)

> **Nota:** Cityscapes tiene 19 clases de evaluación estándar (road, sidewalk, building, wall, fence, pole, traffic light, traffic sign, vegetation, terrain, sky, person, rider, car, truck, bus, train, motorcycle, bicycle).

In [ ]:
import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.models import ResNet50_Weights
import torchmetrics
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibilidad ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Dispositivo ─────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Dispositivo: {DEVICE}')

# ── Hiperparámetros del paper (Sección 4.1) ─────────────────────────────────
NUM_CLASSES    = 19          # Clases de evaluación de Cityscapes
K_CLUSTERS     = 3           # Número de clusters por clase (K en el paper)
CAM_THRESHOLD  = 0.7         # Umbral τ para features confiables (Sección 3.1)
THETA          = 3.0         # Peso ϑ de L_otcon en la loss total (Eq. 15)
LR_ENCODER     = 0.1         # Learning rate del encoder
LR_EXTRA       = 1.0         # Learning rate de capas adicionales
BATCH_SIZE     = 4           # Reducido para Colab (paper usa 16 con más GPU)
NUM_EPOCHS     = 6           # Épocas de entrenamiento (igual al paper)
IMG_SIZE       = 224         # Tamaño de entrada (reducido para Colab)
FEAT_DIM       = 2048        # Dimensión de features de ResNet50 (capa 4)
SINKHORN_REG   = 0.1         # Regularización para Sinkhorn OT
SINKHORN_ITERS = 100         # Iteraciones del algoritmo Sinkhorn

# ── Rutas de Cityscapes (ajustar según tu Google Drive) ─────────────────────
DRIVE_ROOT     = '/content/drive/MyDrive/datasets/cityscapes_raw'
DATA_ROOT  = '/content/cityscapes/cityscapes'
IMG_SUBDIR = 'leftImg8bit_trainvaltest/leftImg8bit'
GT_SUBDIR  = 'gtFine_trainvaltest/gtFine'

# ── 19 clases de Cityscapes con sus colores oficiales ───────────────────────
CITYSCAPES_CLASSES = [
    'road', 'sidewalk', 'building', 'wall', 'fence',
    'pole', 'traffic light', 'traffic sign', 'vegetation', 'terrain',
    'sky', 'person', 'rider', 'car', 'truck',
    'bus', 'train', 'motorcycle', 'bicycle'
]

CITYSCAPES_COLORS = [
    (128,64,128), (244,35,232), (70,70,70), (102,102,156), (190,153,153),
    (153,153,153), (250,170,30), (220,220,0), (107,142,35), (152,251,152),
    (70,130,180), (220,20,60), (255,0,0), (0,0,142), (0,0,70),
    (0,60,100), (0,80,100), (0,0,230), (119,11,32)
]

# Mapping de IDs de Cityscapes a índices de clase 0-18
CITYSCAPES_ID_TO_IDX = {
    7:0, 8:1, 11:2, 12:3, 13:4,
    17:5, 19:6, 20:7, 21:8, 22:9,
    23:10, 24:11, 25:12, 26:13, 27:14,
    28:15, 31:16, 32:17, 33:18
}

print(f'✅ Configuración lista — {NUM_CLASSES} clases, K={K_CLUSTERS} clusters')

🖥️  Dispositivo: cpu
✅ Configuración lista — 19 clases, K=3 clusters


---
## 📁 SECCIÓN 2 — Montar Drive y descomprimir Cityscapes

Montamos Google Drive donde tienes los 3 archivos zip:
- `leftImg8bit_trainvaltest.zip` → imágenes RGB
- `gtFine_trainvaltest.zip` → anotaciones finas (train/val)
- `gtCoarse.zip` → anotaciones gruesas (opcional, para training extra)

**Solo descomprimimos si no existe ya el directorio** para no repetir el proceso.

In [ ]:
from google.colab import drive

# ── Montar Google Drive ──────────────────────────────────────────────────────
drive.mount('/content/drive')
print('✅ Drive montado')

# ── Crear directorio destino ─────────────────────────────────────────────────
os.makedirs(DATA_ROOT, exist_ok=True)

def unzip_if_needed(zip_name, check_path):
    """Descomprime solo si el directorio destino no existe."""
    zip_path = os.path.join(DRIVE_ROOT, zip_name)
    if not os.path.exists(check_path):
        print(f'📦 Descomprimiendo {zip_name}...')
        !unzip -q "{zip_path}" -d "{DATA_ROOT}"
        print(f'   ✅ Listo: {check_path}')
    else:
        print(f'   ⏭️  Ya existe: {check_path}')

# Descomprimir imágenes y anotaciones finas
unzip_if_needed(
    'leftImg8bit_trainvaltest.zip',
    os.path.join(DATA_ROOT, 'leftImg8bit')
)
unzip_if_needed(
    'gtFine_trainvaltest.zip',
    os.path.join(DATA_ROOT, 'gtFine')
)

# Verificar estructura
print('\n📂 Estructura del dataset:')
for split in ['train', 'val']:
    img_path = os.path.join(DATA_ROOT, 'leftImg8bit_trainvaltest', 'leftImg8bit', split)
    gt_path  = os.path.join(DATA_ROOT, 'gtFine_trainvaltest', 'gtFine', split)
    if os.path.exists(img_path):
        cities = os.listdir(img_path)
        n_imgs = sum(len(os.listdir(os.path.join(img_path, c))) for c in cities)
        print(f'   {split}: {n_imgs} imágenes en {len(cities)} ciudades')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado
📦 Descomprimiendo leftImg8bit_trainvaltest.zip...
replace /content/cityscapes/cityscapes/__MACOSX/._leftImg8bit_trainvaltest? [y]es, [n]o, [A]ll, [N]one, [r]ename: All
y
yes


   ✅ Listo: /content/cityscapes/cityscapes/leftImg8bit
📦 Descomprimiendo gtFine_trainvaltest.zip...
replace /content/cityscapes/cityscapes/__MACOSX/._gtFine_trainvaltest? [y]es, [n]o, [A]ll, [N]one, [r]ename:    ✅ Listo: /content/cityscapes/cityscapes/gtFine

📂 Estructura del dataset:
   train: 2975 imágenes en 18 ciudades
   val: 500 imágenes en 3 ciudades


In [ ]:
import os

# ── Ver qué hay en /content/cityscapes ──────────────────────────────────────
print("📂 Contenido de /content/cityscapes:")
for root, dirs, files in os.walk('/content/cityscapes'):
    level = root.replace('/content/cityscapes', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:  # solo 3 niveles de profundidad
        for f in files[:3]:
            print(f'{indent}  {f}')
    if level == 2:
        break

📂 Contenido de /content/cityscapes:
cityscapes/
  cityscapes/
    __MACOSX/
      ._leftImg8bit_trainvaltest
      ._gtFine_trainvaltest


---
## 🗃️ SECCIÓN 3 — Dataset y DataLoader de Cityscapes

Implementamos un Dataset personalizado que:
1. Carga imágenes RGB (`leftImg8bit`) y máscaras de ground truth (`gtFine_labelIds`)
2. Convierte los IDs originales de Cityscapes a índices 0-18
3. Genera **etiquetas de imagen** (image-level labels) a partir de las clases presentes en la máscara
   — esto simula la supervisión débil: en lugar de usar la máscara pixel a pixel durante el entrenamiento,
   solo usamos qué clases están presentes en la imagen.

> **Supervisión débil:** El modelo solo recibe un vector binario indicando qué clases hay en la imagen,
> NO la máscara de segmentación. Las máscaras solo se usan para **evaluar** la calidad de los CAMs.

In [ ]:
class CityscapesWeakDataset(Dataset):
    """
    Dataset de Cityscapes para entrenamiento con supervisión débil.
    Durante el entrenamiento: solo usa etiquetas de imagen (qué clases hay).
    Durante la evaluación: usa las máscaras ground truth para calcular mIoU.
    """
    def __init__(self, root, split='train', transform=None, mask_transform=None):
        self.root           = root
        self.split          = split
        self.transform      = transform
        self.mask_transform = mask_transform
        self.id_to_idx      = CITYSCAPES_ID_TO_IDX
        self.samples        = []
        self._build_file_list()

    def _build_file_list(self):
        img_dir = os.path.join(self.root, 'leftImg8bit_trainvaltest', 'leftImg8bit', self.split)
        gt_dir  = os.path.join(self.root, 'gtFine_trainvaltest', 'gtFine', self.split)
        for city in sorted(os.listdir(img_dir)):
            city_img_dir = os.path.join(img_dir, city)
            city_gt_dir  = os.path.join(gt_dir, city)
            for fname in sorted(os.listdir(city_img_dir)):
                if not fname.endswith('_leftImg8bit.png'):
                    continue
                prefix   = fname.replace('_leftImg8bit.png', '')
                gt_fname = prefix + '_gtFine_labelIds.png'
                img_path = os.path.join(city_img_dir, fname)
                gt_path  = os.path.join(city_gt_dir, gt_fname)
                if os.path.exists(gt_path):
                    self.samples.append((img_path, gt_path))
        print(f'   [{self.split}] {len(self.samples)} muestras cargadas')

    def _label_ids_to_idx(self, mask_pil):
        """Convierte máscara de IDs originales a índices 0-18 (255 = ignore)."""
        mask_np  = np.array(mask_pil, dtype=np.int32)
        out_mask = np.full(mask_np.shape, 255, dtype=np.uint8)
        for orig_id, idx in self.id_to_idx.items():
            out_mask[mask_np == orig_id] = idx
        return out_mask

    def _get_image_label(self, mask_idx):
        """Vector binario [NUM_CLASSES] indicando qué clases están presentes."""
        label = torch.zeros(NUM_CLASSES, dtype=torch.float32)
        unique = np.unique(mask_idx)
        for u in unique:
            if 0 <= u < NUM_CLASSES:
                label[u] = 1.0
        return label

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, gt_path = self.samples[idx]
        image    = Image.open(img_path).convert('RGB')
        mask_pil = Image.open(gt_path)

        # Convertir IDs a índices 0-18
        mask_idx = self._label_ids_to_idx(mask_pil)

        # Etiqueta de imagen (supervisión débil)
        img_label = self._get_image_label(mask_idx)

        # Transformaciones
        if self.transform:
            image = self.transform(image)

        # Máscara para evaluación
        mask_tensor = torch.from_numpy(mask_idx).long()
        if self.mask_transform:
            mask_pil_tmp = Image.fromarray(mask_idx)
            mask_tensor  = torch.from_numpy(
                np.array(mask_pil_tmp.resize((IMG_SIZE, IMG_SIZE), Image.NEAREST))
            ).long()
        else:
            mask_tensor = torch.from_numpy(
                np.array(Image.fromarray(mask_idx).resize(
                    (IMG_SIZE, IMG_SIZE), Image.NEAREST))
            ).long()

        return image, img_label, mask_tensor


# ── Transformaciones ─────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── Crear datasets y dataloaders ─────────────────────────────────────────────
print('Cargando datasets...')
train_dataset = CityscapesWeakDataset(DATA_ROOT, split='train', transform=train_transform)
val_dataset   = CityscapesWeakDataset(DATA_ROOT, split='val',   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'✅ Train: {len(train_dataset)} imágenes | Val: {len(val_dataset)} imágenes')
print(f'   Batches por época: {len(train_loader)}')

Cargando datasets...
   [train] 2975 muestras cargadas
   [val] 500 muestras cargadas
✅ Train: 2975 imágenes | Val: 500 imágenes
   Batches por época: 743


---
## 🏗️ SECCIÓN 4 — Arquitectura: Encoder (ResNet50) + Clasificador + CAM

Implementamos el **Encoder** basado en ResNet50 pre-entrenado en ImageNet (como en el paper).

El encoder expone los features de las 4 capas (F1, F2, F3, F4) necesarios para la proyección
jerárquica de la Eq. (2) y (8) del paper. El clasificador usa Global Average Pooling (GAP)
seguido de una capa lineal para predecir qué clases están presentes (Eq. 16).

Los **CAMs del clasificador** se generan como: `CAM_c = ReLU(W_c^T · F4)` (Eq. 1 del paper).

In [ ]:
class HierarchicalProjection(nn.Module):
    """
    Proyección lineal L(F1, F2, F3, F4) del paper (Eq. 2 y 8).
    Combina features de múltiples capas del encoder en una representación unificada.
    Canales de ResNet50: F1=256, F2=512, F3=1024, F4=2048 → proyecta a FEAT_DIM.
    """
    def __init__(self, out_dim=2048):
        super().__init__()
        # Cada capa se proyecta al espacio de features final
        self.proj1 = nn.Conv2d(256,  out_dim, kernel_size=1, bias=False)
        self.proj2 = nn.Conv2d(512,  out_dim, kernel_size=1, bias=False)
        self.proj3 = nn.Conv2d(1024, out_dim, kernel_size=1, bias=False)
        self.proj4 = nn.Conv2d(2048, out_dim, kernel_size=1, bias=False)
        self.bn    = nn.BatchNorm2d(out_dim)
        self.relu  = nn.ReLU(inplace=True)

    def forward(self, f1, f2, f3, f4):
        H, W = f4.shape[2], f4.shape[3]
        p1 = F.interpolate(self.proj1(f1), size=(H, W), mode='bilinear', align_corners=False)
        p2 = F.interpolate(self.proj2(f2), size=(H, W), mode='bilinear', align_corners=False)
        p3 = F.interpolate(self.proj3(f3), size=(H, W), mode='bilinear', align_corners=False)
        p4 = self.proj4(f4)
        return self.relu(self.bn(p1 + p2 + p3 + p4))


class POTEncoder(nn.Module):
    """
    Encoder basado en ResNet50 preentrenado en ImageNet.
    Expone F1, F2, F3, F4 para la proyección jerárquica.
    Incluye el clasificador multi-label y la generación de CAMs.
    """
    def __init__(self, num_classes=NUM_CLASSES, feat_dim=FEAT_DIM):
        super().__init__()
        backbone = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

        # ── Extraer capas del backbone ───────────────────────────────────────
        self.layer0 = nn.Sequential(backbone.conv1, backbone.bn1,
                                    backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1   # → [B, 256,  H/4,  W/4]
        self.layer2 = backbone.layer2   # → [B, 512,  H/8,  W/8]
        self.layer3 = backbone.layer3   # → [B, 1024, H/16, W/16]
        self.layer4 = backbone.layer4   # → [B, 2048, H/32, W/32] = F4

        # ── Proyección jerárquica L(F1,F2,F3,F4) ────────────────────────────
        self.hier_proj = HierarchicalProjection(out_dim=feat_dim)

        # ── Clasificador: peso ω_c para cada clase (Eq. 1 y 16) ─────────────
        self.classifier = nn.Linear(feat_dim, num_classes, bias=False)

        self.num_classes = num_classes
        self.feat_dim    = feat_dim

    def forward(self, x):
        """
        Returns:
            logits     : [B, C] predicciones de clasificación
            cam_cls    : [B, C+1, H, W] CAMs del clasificador (incluyendo background)
            feats      : [B, D, H, W] features jerárquicas proyectadas
            f1,f2,f3,f4: features crudas por capa
        """
        # ── Forward por el backbone ──────────────────────────────────────────
        x  = self.layer0(x)
        f1 = self.layer1(x)    # [B, 256,  H/4,  W/4]
        f2 = self.layer2(f1)   # [B, 512,  H/8,  W/8]
        f3 = self.layer3(f2)   # [B, 1024, H/16, W/16]
        f4 = self.layer4(f3)   # [B, 2048, H/32, W/32]

        # ── Proyección jerárquica ────────────────────────────────────────────
        feats = self.hier_proj(f1, f2, f3, f4)   # [B, D, H, W]

        # ── Clasificador (GAP + linear) ──────────────────────────────────────
        gap    = F.adaptive_avg_pool2d(feats, 1).flatten(1)  # [B, D]
        logits = self.classifier(gap)                         # [B, C]

        # ── CAMs del clasificador: ReLU(W_c^T · feats) (Eq. 1) ──────────────
        W = self.classifier.weight   # [C, D]
        # Foreground CAMs: [B, C, H, W]
        cam_fg = F.relu(
            torch.einsum('cd,bdhw->bchw', W, feats)
        )
        # Normalizar cada CAM a [0, 1]
        cam_max = cam_fg.flatten(2).max(dim=2)[0].unsqueeze(-1).unsqueeze(-1) + 1e-8
        cam_fg  = cam_fg / cam_max

        # Background CAM: 1 - max_c(cam_fg_c) (Eq. 1 del paper)
        cam_bg  = 1.0 - cam_fg.max(dim=1, keepdim=True)[0]  # [B, 1, H, W]

        # CAM completo con background como clase 0
        cam_cls = torch.cat([cam_bg, cam_fg], dim=1)          # [B, C+1, H, W]

        return logits, cam_cls, feats, (f1, f2, f3, f4)


# ── Instanciar modelo ────────────────────────────────────────────────────────
model = POTEncoder(num_classes=NUM_CLASSES, feat_dim=FEAT_DIM).to(DEVICE)

# Parámetros: separamos encoder de capas adicionales
encoder_params = list(model.layer0.parameters()) + list(model.layer1.parameters()) + \
                 list(model.layer2.parameters()) + list(model.layer3.parameters()) + \
                 list(model.layer4.parameters())
extra_params   = list(model.hier_proj.parameters()) + list(model.classifier.parameters())

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✅ Modelo POTEncoder creado — {total_params:.1f}M parámetros')

# Verificar con un batch dummy
with torch.no_grad():
    dummy   = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    logits, cam_cls, feats, layers = model(dummy)
    print(f'   logits: {logits.shape}  cam_cls: {cam_cls.shape}  feats: {feats.shape}')

✅ Modelo POTEncoder creado — 31.4M parámetros
   logits: torch.Size([2, 19])  cam_cls: torch.Size([2, 20, 7, 7])  feats: torch.Size([2, 2048, 7, 7])


---
## 🔵 SECCIÓN 5 — POT Stage I: Feature Partitioning (K-Means + Prototipos)

Esta etapa implementa la **Sección 3.1** del paper:

1. Para cada clase presente en la imagen, seleccionamos los features con **CAM > τ** (features confiables)
2. Aplicamos **K-Means** con K=3 clusters sobre esos features confiables
3. Generamos un **prototipo** por cluster via Mask Average Pooling (MAP): `P = MAP(Mask · L(F1,F2,F3,F4))`
4. También generamos el **prototipo del clasificador CAM** `P_cls` (un solo prototipo global por clase)
   que se usará como referencia en la constraint del OT (Eq. 4 y 5)

El resultado es: `P ∈ R^{(J+1) × K × D}` — prototipos de cluster para J clases + background

In [ ]:
def mask_average_pool(features, mask):
    """
    Mask Average Pooling: MAP(Mask · Features)
    features: [D, H*W]  — features espaciales aplanadas
    mask    : [H*W]     — máscara binaria de la región
    Returns : [D]       — prototipo de la región
    """
    mask = mask.float()
    if mask.sum() < 1:
        return features.mean(dim=1)   # fallback: media global
    return (features * mask.unsqueeze(0)).sum(dim=1) / mask.sum()


def feature_partitioning(feats_flat, cam_cls, img_label, K=K_CLUSTERS, threshold=CAM_THRESHOLD):
    """
    Etapa I del paper (Sección 3.1): Feature Partitioning.

    Args:
        feats_flat : [D, HW]      features proyectadas aplanadas
        cam_cls    : [C+1, HW]    CAMs del clasificador (incluyendo background como clase 0)
        img_label  : [C]          etiqueta de imagen (qué clases están presentes)
        K          : int          número de clusters por clase
        threshold  : float        umbral τ para features confiables

    Returns:
        cluster_protos : [J+1, K, D]  prototipos de cluster (J clases + 1 background)
        cls_protos     : [J+1, D]     prototipos del clasificador CAM (1 por clase)
        cluster_masks  : [HW, J+1, K] máscara binaria de asignación inicial
        present_classes: list         índices de clases presentes (incluyendo background=0)
    """
    D, HW    = feats_flat.shape
    C        = img_label.shape[0]           # Número de clases foreground
    J1       = C + 1                        # Foreground + background
    feats_np = feats_flat.T.cpu().numpy()   # [HW, D] para sklearn

    cluster_protos = torch.zeros(J1, K, D, device=feats_flat.device)
    cls_protos     = torch.zeros(J1, D,    device=feats_flat.device)
    cluster_masks  = torch.zeros(HW, J1, K, device=feats_flat.device)
    present_classes = []

    # ── Background (clase 0) ─────────────────────────────────────────────────
    bg_cam    = cam_cls[0]                           # [HW]
    bg_conf   = (bg_cam > threshold).cpu().numpy()   # features confiables del bg
    # Prototipo del clasificador para background
    bg_mask_t = torch.from_numpy(bg_conf.astype(np.float32)).to(feats_flat.device)
    cls_protos[0] = mask_average_pool(feats_flat, bg_mask_t)

    if bg_conf.sum() >= K:
        try:
            km    = KMeans(n_clusters=K, n_init=3, random_state=SEED)
            labels = km.fit_predict(feats_np[bg_conf])
            conf_indices = np.where(bg_conf)[0]
            for k in range(K):
                k_indices = conf_indices[labels == k]
                if len(k_indices) > 0:
                    k_mask = torch.zeros(HW, device=feats_flat.device)
                    k_mask[k_indices] = 1.0
                    cluster_protos[0, k] = mask_average_pool(feats_flat, k_mask)
                    cluster_masks[:, 0, k] = k_mask
        except Exception:
            for k in range(K):
                cluster_protos[0, k] = cls_protos[0]
    else:
        for k in range(K):
            cluster_protos[0, k] = cls_protos[0]

    present_classes.append(0)

    # ── Clases foreground presentes ──────────────────────────────────────────
    fg_present = torch.where(img_label > 0.5)[0].tolist()

    for c in fg_present:
        class_idx  = c + 1                         # índice en cam_cls (bg=0)
        cam_c      = cam_cls[class_idx]             # [HW]
        conf_mask  = (cam_c > threshold).cpu().numpy()

        # Prototipo del clasificador para clase c (único, para la constraint)
        conf_t = torch.from_numpy(conf_mask.astype(np.float32)).to(feats_flat.device)
        cls_protos[class_idx] = mask_average_pool(feats_flat, conf_t)

        if conf_mask.sum() >= K:
            try:
                km     = KMeans(n_clusters=K, n_init=3, random_state=SEED)
                labels = km.fit_predict(feats_np[conf_mask])
                conf_indices = np.where(conf_mask)[0]
                for k in range(K):
                    k_indices = conf_indices[labels == k]
                    if len(k_indices) > 0:
                        k_mask = torch.zeros(HW, device=feats_flat.device)
                        k_mask[k_indices] = 1.0
                        cluster_protos[class_idx, k] = mask_average_pool(feats_flat, k_mask)
                        cluster_masks[:, class_idx, k] = k_mask
            except Exception:
                for k in range(K):
                    cluster_protos[class_idx, k] = cls_protos[class_idx]
        else:
            # Pocos features: repetir el prototipo único
            for k in range(K):
                cluster_protos[class_idx, k] = cls_protos[class_idx]

        present_classes.append(class_idx)

    return cluster_protos, cls_protos, cluster_masks, present_classes


print('✅ Feature Partitioning implementado')
print(f'   Salida: cluster_protos=[J+1={NUM_CLASSES+1}, K={K_CLUSTERS}, D={FEAT_DIM}]')

✅ Feature Partitioning implementado
   Salida: cluster_protos=[J+1=20, K=3, D=2048]


---
## 🔴 SECCIÓN 6 — POT Stage II: Feature Allocation (Similarity-Aware Optimal Transport)

Esta etapa implementa la **Sección 3.2** del paper — el corazón del método.

**Problema estándar de OT** (Eq. 3): encontrar el plan de transporte T que minimiza el costo
de mover masa de la distribución de features a la distribución de prototipos.

**Innovación del paper** (Eq. 5-6): en lugar de usar distribución uniforme en el target,
construimos una **constraint similarity-aware** `x = softmax(cosine_sim(P_cluster, P_cls))`
que prioriza los prototipos de cluster más similares al prototipo del clasificador.

Se resuelve con el **algoritmo de Sinkhorn** (OT regularizado) para eficiencia computacional.

Resultado: `T* ∈ R^{HW × (J+1)×K}` — plan de transporte que indica con qué probabilidad
cada feature pertenece a cada cluster.

In [ ]:
def cosine_similarity_matrix(A, B, eps=1e-8):
    """
    Calcula la matriz de similitud coseno entre filas de A y B.
    A: [M, D]  B: [N, D]  → Retorna [M, N]
    """
    A_norm = F.normalize(A.float(), dim=1, eps=eps)
    B_norm = F.normalize(B.float(), dim=1, eps=eps)
    return torch.mm(A_norm, B_norm.T)


def compute_similarity_aware_constraint(cluster_protos, cls_protos, present_classes):
    """
    Construye la restricción similarity-aware del target (Eq. 4-5 del paper).

    Para cada clase c y cluster k:
        x_{c,k} = cos_sim(P_{c,k}, P^cls_c)
    Luego: x = softmax(flatten(X))

    Args:
        cluster_protos : [J+1, K, D]
        cls_protos     : [J+1, D]
        present_classes: lista de índices de clases presentes

    Returns:
        x_constraint: [N] donde N = len(present_classes) * K
                      distribución normalizada del target (Eq. 5)
    """
    sims = []
    for c_idx in present_classes:
        p_cls = cls_protos[c_idx].unsqueeze(0)          # [1, D]
        p_clust = cluster_protos[c_idx]                  # [K, D]
        sim_ck = cosine_similarity_matrix(p_clust, p_cls).squeeze(1)  # [K]
        sims.append(sim_ck)

    X_flat = torch.cat(sims, dim=0)       # [N] = [len(present)*K]
    x      = F.softmax(X_flat, dim=0)    # Eq. 5: normalización
    return x


def sinkhorn_ot(S, x_constraint, reg=SINKHORN_REG, n_iter=SINKHORN_ITERS):
    """
    Sinkhorn OT implementado manualmente — sin librería POT.
    Matemáticamente equivalente a ot.sinkhorn().
    """
    M_cost = (1.0 - S).clamp(0, 2).float()
    K      = torch.exp(-M_cost / reg)        # Kernel: [HW, N]

    u = torch.full((S.shape[0],), 1.0 / S.shape[0], device=S.device)
    v = x_constraint.to(S.device).float()

    a = torch.ones(S.shape[0], device=S.device)
    b = torch.ones(len(x_constraint), device=S.device)

    for _ in range(n_iter):
        a = u / (K @ b + 1e-9)
        b = v / (K.T @ a + 1e-9)

    T_star = a.unsqueeze(1) * K * b.unsqueeze(0)
    return T_star


def feature_allocation(feats_flat, cluster_protos, cls_protos, present_classes):
    """
    Etapa II del paper (Sección 3.2): Feature Allocation con OT.

    Args:
        feats_flat     : [D, HW] features proyectadas
        cluster_protos : [J+1, K, D] prototipos de cluster
        cls_protos     : [J+1, D] prototipos del clasificador
        present_classes: lista de clases presentes

    Returns:
        T_star       : [HW, N] plan de transporte (N = len(present)*K)
        assignment   : [HW] índice del prototipo asignado a cada feature (Eq. 7)
        x_constraint : [N] restricción similarity-aware
    """
    D, HW = feats_flat.shape
    feats_T = feats_flat.T.float()   # [HW, D]

    # Construir matriz de prototipos para el OT: [N, D]
    proto_list = []
    for c_idx in present_classes:
        proto_list.append(cluster_protos[c_idx])    # [K, D]
    all_protos = torch.cat(proto_list, dim=0)       # [N, D]

    # Matriz de similitudes S [HW, N] (Eq. 6)
    S = cosine_similarity_matrix(feats_T, all_protos)

    # Restricción similarity-aware (Eq. 5)
    x_constraint = compute_similarity_aware_constraint(
        cluster_protos, cls_protos, present_classes
    )

    # Sinkhorn OT (Eq. 6)
    T_star = sinkhorn_ot(S, x_constraint)

    # Asignación greedy (Eq. 7): j* = argmax_j T*_{i,j}
    assignment = T_star.argmax(dim=1)   # [HW]

    return T_star, assignment, x_constraint


print('✅ Feature Allocation (Similarity-Aware OT) implementado')
print(f'   Sinkhorn: reg={SINKHORN_REG}, iters={SINKHORN_ITERS}')

✅ Feature Allocation (Similarity-Aware OT) implementado
   Sinkhorn: reg=0.1, iters=100


---
## 🟢 SECCIÓN 7 — POT Stage III: Feature Activation + OT Consistency Loss

Esta etapa implementa la **Sección 3.3** del paper:

**Reweighting Prototype CAMs:**
1. Re-generar prototipos `P*` usando TODOS los features asignados a cada cluster (Eq. 8)
2. Calcular CAM de cada cluster: `M^p_{c,k} = ReLU(cos_sim(P*_{c,k}, L(F)))` (Eq. 9)
3. Reponderar: `M^{p→}_{c,k} = T*_{c,k} ⊙ M^p_{c,k}` (Eq. 10) — el plan OT como peso
4. Combinar: `M^{p→}_c = Σ_k M^{p→}_{c,k}` (Eq. 11)

**OT Consistency Loss (Eq. 12-14):**
Loss adaptativa que penaliza divergencia entre CAMs prototípicos y CAMs del clasificador,
con pesos por pixel `w_{i,c}` basados en la certeza del plan de transporte (entropía binaria).

In [ ]:
def feature_activation(feats_flat, cluster_protos, T_star, assignment,
                        present_classes, H, W, K=K_CLUSTERS):
    """
    Etapa III del paper (Sección 3.3): Feature Activation.

    Args:
        feats_flat     : [D, HW] features proyectadas
        cluster_protos : [J+1, K, D] prototipos de cluster iniciales
        T_star         : [HW, N] plan de transporte óptimo
        assignment     : [HW] asignación greedy de cada feature
        present_classes: lista de clases presentes
        H, W           : dimensiones espaciales del feature map

    Returns:
        cam_proto: [C+1, H, W] CAMs prototípicos finales (incluyendo background)
        T_star_restructured: plan de transporte organizado por clase
    """
    D, HW     = feats_flat.shape
    J1        = cluster_protos.shape[0]    # J+1 (clases + background)
    C         = J1 - 1                     # clases foreground
    feats_T   = feats_flat.T.float()       # [HW, D]

    # ── Re-generar prototipos con todos los features asignados (Eq. 8) ────────
    updated_protos = cluster_protos.clone()
    N_protos = len(present_classes) * K
    proto_idx = 0

    for c_pos, c_idx in enumerate(present_classes):
        for k in range(K):
            # Seleccionar features asignados a este prototipo
            assign_mask = (assignment == proto_idx).float()   # [HW]
            updated_protos[c_idx, k] = mask_average_pool(feats_flat, assign_mask)
            proto_idx += 1

    # ── Calcular CAMs por cluster (Eq. 9) ────────────────────────────────────
    # cam_proto_full: [J+1, H, W] — se va llenando por clase
    cam_proto_full = torch.zeros(J1, H, W, device=feats_flat.device)
    proto_idx = 0

    for c_pos, c_idx in enumerate(present_classes):
        cam_c_weighted = torch.zeros(HW, device=feats_flat.device)

        for k in range(K):
            # Cosine similarity entre prototipo actualizado y todos los features
            p_ck   = updated_protos[c_idx, k].unsqueeze(0)   # [1, D]
            sim_ck = cosine_similarity_matrix(feats_T, p_ck).squeeze(1)  # [HW]
            cam_ck = F.relu(sim_ck)                           # Eq. 9: ReLU

            # Pesos del plan de transporte para este prototipo (Eq. 10)
            t_ck          = T_star[:, proto_idx]              # [HW]
            cam_ck_weighted = t_ck * cam_ck                   # Eq. 10: reweighting
            cam_c_weighted += cam_ck_weighted                 # Eq. 11: suma
            proto_idx += 1

        # Reshape a espacial
        cam_proto_full[c_idx] = cam_c_weighted.reshape(H, W)

    return cam_proto_full, updated_protos


def binary_entropy(p, eps=1e-8):
    """Entropía binaria H(p) = -p*log(p) - (1-p)*log(1-p) (Eq. 14)"""
    p   = p.clamp(eps, 1 - eps)
    return -p * torch.log(p) - (1 - p) * torch.log(1 - p)


def ot_consistency_loss(cam_proto, cam_cls, T_star, img_label,
                        present_classes, K=K_CLUSTERS):
    """
    OT-based Consistency Loss (Eq. 12-14 del paper).

    Penaliza la divergencia entre CAMs prototípicos y CAMs del clasificador,
    con pesos adaptativos w_{i,c} basados en la certeza del plan OT.

    Args:
        cam_proto : [J+1, HW] CAMs prototípicos aplanados
        cam_cls   : [J+1, HW] CAMs del clasificador aplanados
        T_star    : [HW, N] plan de transporte
        img_label : [C] etiqueta de imagen
        present_classes: lista de clases presentes

    Returns:
        loss_otcon: escalar
    """
    HW       = cam_proto.shape[1]
    J1       = cam_proto.shape[0]
    C        = J1 - 1
    log_C1   = np.log(C + 1) + 1e-8    # Factor de escala 1/log(|C|+1)

    # Normalizar CAMs (ε1 normalización por clase para cada pixel)
    cam_proto_norm = F.normalize(cam_proto.float(), p=1, dim=0)
    cam_cls_norm   = F.normalize(cam_cls.float(),   p=1, dim=0)

    total_loss  = torch.tensor(0.0, device=cam_proto.device, requires_grad=False)
    proto_idx   = 0
    count       = 0

    for c_pos, c_idx in enumerate(present_classes):
        # g(t_{i,c}) = Σ_k t_{i,c,k} — prob de que feature i pertenezca a clase c (Eq. 13)
        t_ic = T_star[:, proto_idx:proto_idx+K].sum(dim=1)   # [HW]
        proto_idx += K

        # Entropía binaria (Eq. 14)
        H_t   = binary_entropy(t_ic)                          # [HW]

        # Peso adaptativo w_{i,c} (Eq. 13)
        # l_c = 1 si la clase está presente (one-hot), 0 si no
        l_c   = 1.0 if c_idx == 0 else float(img_label[c_idx - 1].item())
        w_ic  = l_c * (1.0 - H_t / log_C1)                   # [HW]
        w_ic  = w_ic.clamp(0)                                 # ≥ 0

        # L1 entre CAMs normalizados ponderado por w_{i,c}
        diff  = (cam_proto_norm[c_idx] - cam_cls_norm[c_idx]).abs()  # [HW]
        loss_c = (w_ic * diff).sum() / (HW + 1e-8)
        total_loss = total_loss + loss_c
        count += 1

    return total_loss / max(count, 1)


print('✅ Feature Activation y OT Consistency Loss implementados')

✅ Feature Activation y OT Consistency Loss implementados


---
## 🔧 SECCIÓN 8 — Loss de clasificación y función POT completa

Aquí integramos todo el pipeline en una función `pot_forward` que:
1. Recibe un batch de imágenes y etiquetas débiles
2. Corre el encoder para obtener CAMs y features
3. Ejecuta las 3 etapas de POT imagen por imagen
4. Calcula la **loss total**: `L_total = L_cls + ϑ * L_otcon` (Eq. 15)

La **loss de clasificación** (Eq. 16) es multi-label soft margin:
`L_cls = (1/|C|) Σ_c [l_c * log(σ(GAP(ω_c^T · F))) + (1-l_c) * log(1 - σ(...))]`

In [ ]:
def classification_loss(logits, img_labels):
    """
    Multi-label soft margin loss (Eq. 16 del paper).
    logits    : [B, C]
    img_labels: [B, C] — etiquetas binarias de imagen
    """
    return F.multilabel_soft_margin_loss(logits, img_labels)


def pot_forward_single(feats, cam_cls, img_label, H, W):
    """
    Ejecuta el pipeline completo de POT para UNA imagen.

    Returns:
        cam_proto_full: [J+1, H, W] CAMs prototípicos finales
        loss_otcon    : escalar de consistency loss
    """
    D = feats.shape[0]
    # Aplanar: [D, HW]
    feats_flat = feats.reshape(D, -1)
    cam_flat   = cam_cls.reshape(cam_cls.shape[0], -1)   # [C+1, HW]

    # ── Stage I: Feature Partitioning ────────────────────────────────────────
    cluster_protos, cls_protos, _, present_classes = feature_partitioning(
        feats_flat, cam_flat, img_label
    )

    # ── Stage II: Feature Allocation (Sinkhorn OT) ───────────────────────────
    T_star, assignment, x_constraint = feature_allocation(
        feats_flat, cluster_protos, cls_protos, present_classes
    )

    # ── Stage III: Feature Activation ────────────────────────────────────────
    cam_proto_full, _ = feature_activation(
        feats_flat, cluster_protos, T_star, assignment,
        present_classes, H, W
    )

    # ── OT Consistency Loss ──────────────────────────────────────────────────
    cam_proto_flat = cam_proto_full.reshape(cam_proto_full.shape[0], -1)
    loss_otcon = ot_consistency_loss(
        cam_proto_flat, cam_flat, T_star, img_label, present_classes
    )

    return cam_proto_full, loss_otcon


def cam_to_segmentation(cam_proto, threshold=0.3):
    """
    Convierte CAMs prototípicos a máscara de segmentación.
    cam_proto: [J+1, H, W] — clase 0 es background
    Returns  : [H, W] — mapa de clases (0=background, 1..C=foreground)
    """
    # Normalizar
    cam_norm = cam_proto.clone()
    for c in range(cam_norm.shape[0]):
        mx = cam_norm[c].max()
        if mx > 0:
            cam_norm[c] /= mx

    # Asignar cada pixel a la clase con mayor activación
    seg_map = cam_norm.argmax(dim=0)   # [H, W] — índices 0=bg, 1..C=fg

    # Si la activación máxima es muy baja, marcar como background
    max_act  = cam_norm.max(dim=0)[0]
    seg_map[max_act < threshold] = 0

    return seg_map


print('✅ Pipeline POT completo implementado')
print('   L_total = L_cls + ϑ * L_otcon  (ϑ =', THETA, ')')

✅ Pipeline POT completo implementado
   L_total = L_cls + ϑ * L_otcon  (ϑ = 3.0 )


---
## 🚀 SECCIÓN 9 — Entrenamiento

Loop de entrenamiento completo con:
- **Optimizer SGD** con momentum (como en el paper, Sección 4.1)
- **Learning rate scheduler** polinomial con power=0.9
- **Tasas diferenciadas**: `LR=0.1` para el encoder, `LR=1.0` para capas extra
- Guardado del mejor modelo según mIoU de validación
- Logging de losses por época

> ⚠️ El paper entrena 6 épocas con batch=16 y GPU completa. En Colab con batch=4
> y T4, el tiempo por época puede ser ~20-40 min. Se recomienda usar el **subset de entrenamiento**
> o reducir `SUBSET_SIZE` si quieres una prueba rápida.

In [ ]:
# ── Opcional: subset para prueba rápida (poner None para dataset completo) ───
SUBSET_SIZE = 500   # None = dataset completo | int = primeras N imágenes

if SUBSET_SIZE is not None:
    from torch.utils.data import Subset
    train_sub    = Subset(train_dataset, range(min(SUBSET_SIZE, len(train_dataset))))
    train_loader = DataLoader(train_sub, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True, drop_last=True)
    print(f'⚡ Usando subset de {SUBSET_SIZE} imágenes para entrenamiento rápido')

# ── Optimizer SGD con tasas diferenciadas (Sección 4.1) ─────────────────────
optimizer = torch.optim.SGD([
    {'params': encoder_params, 'lr': LR_ENCODER},
    {'params': extra_params,   'lr': LR_EXTRA}
], momentum=0.9, weight_decay=1e-4)

total_steps = NUM_EPOCHS * len(train_loader)

def polynomial_lr(step, total_steps, power=0.9, base_lr=LR_ENCODER):
    """Learning rate scheduler polinomial — con protección contra valores negativos."""
    factor = max(0.0, 1 - step / total_steps)
    return float(base_lr * (factor ** power))  # float() garantiza que no sea complejo

# ── Historial de métricas ────────────────────────────────────────────────────
history = {
    'train_loss_cls': [], 'train_loss_otcon': [], 'train_loss_total': [],
    'val_miou': [], 'val_dice': [], 'val_pixel_acc': []
}

best_miou   = 0.0
best_ckpt   = '/content/pot_best_model.pth'
global_step = 0


def train_one_epoch(epoch):
    global global_step
    model.train()
    total_loss_cls   = 0.0
    total_loss_otcon = 0.0
    n_batches        = len(train_loader)

    for batch_idx, (images, img_labels, _) in enumerate(train_loader):
        images     = images.to(DEVICE)
        img_labels = img_labels.to(DEVICE)

        # ── Actualizar LR (scheduler polinomial) ────────────────────────────
        lr_enc = float(polynomial_lr(global_step, total_steps))
        lr_ext = float(lr_enc * (LR_EXTRA / LR_ENCODER))
        optimizer.param_groups[0]['lr'] = lr_enc
        optimizer.param_groups[1]['lr'] = lr_ext

        # ── Forward del encoder ─────────────────────────────────────────────
        logits, cam_cls, feats, _ = model(images)

        # ── Loss de clasificación (Eq. 16) ───────────────────────────────────
        loss_cls = classification_loss(logits, img_labels)

        # ── POT: procesar imagen por imagen (Stage I, II, III) ───────────────
        B, _, H, W = feats.shape
        batch_loss_otcon = torch.tensor(0.0, device=DEVICE)

        for b in range(B):
            feat_b  = feats[b].detach()       # [D, H, W] — sin gradiente en OT
            cam_b   = cam_cls[b].detach()     # [C+1, H, W]
            label_b = img_labels[b]           # [C]

            try:
                _, loss_otcon_b = pot_forward_single(feat_b, cam_b, label_b, H, W)
                batch_loss_otcon = batch_loss_otcon + loss_otcon_b
            except Exception as e:
                pass   # Si falla el OT, solo usamos L_cls

        batch_loss_otcon = batch_loss_otcon / B

        # ── Loss total (Eq. 15) ──────────────────────────────────────────────
        loss_total = loss_cls + THETA * batch_loss_otcon

        # ── Backward y optimizer step ────────────────────────────────────────
        optimizer.zero_grad()
        loss_total.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        global_step += 1

        total_loss_cls   += loss_cls.item()
        total_loss_otcon += batch_loss_otcon.item()

        if batch_idx % 50 == 0:
            print(f'   Época {epoch+1} [{batch_idx}/{n_batches}] '
                  f'L_cls={loss_cls.item():.4f} '
                  f'L_otcon={batch_loss_otcon.item():.4f} '
                  f'L_total={loss_total.item():.4f} '
                  f'LR={lr_enc:.5f}')

    avg_cls   = total_loss_cls   / n_batches
    avg_otcon = total_loss_otcon / n_batches
    avg_total = avg_cls + THETA * avg_otcon
    history['train_loss_cls'].append(avg_cls)
    history['train_loss_otcon'].append(avg_otcon)
    history['train_loss_total'].append(avg_total)
    return avg_cls, avg_otcon, avg_total


print('✅ Loop de entrenamiento configurado')
print(f'   Épocas: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | Pasos totales: {total_steps}')

⚡ Usando subset de 500 imágenes para entrenamiento rápido
✅ Loop de entrenamiento configurado
   Épocas: 6 | Batch: 4 | Pasos totales: 750


---
## 📊 SECCIÓN 10 — Evaluación: mIoU, Dice y Pixel Accuracy

Evaluamos la calidad de los CAMs generados comparándolos con las máscaras ground truth.
Las métricas reportadas son:

- **mIoU** (mean Intersection over Union): métrica principal de Cityscapes
- **Dice coefficient**: 2*|A∩B|/(|A|+|B|) — complementa el IoU
- **Pixel Accuracy**: porcentaje de pixels correctamente clasificados

Para cada imagen, convertimos los CAMs prototípicos a una máscara de segmentación
y la comparamos con el ground truth (ignorando pixels con label=255).

In [ ]:
def evaluate(loader, n_samples=None):
    """
    Evalúa el modelo en mIoU, Dice y Pixel Accuracy.

    Args:
        loader   : DataLoader de validación
        n_samples: limitar a N imágenes (None = todas)

    Returns:
        dict con métricas promedio
    """
    model.eval()

    # Acumuladores por clase para IoU y Dice
    intersection = np.zeros(NUM_CLASSES)
    union        = np.zeros(NUM_CLASSES)
    dice_num     = np.zeros(NUM_CLASSES)
    dice_den     = np.zeros(NUM_CLASSES)
    total_correct = 0
    total_pixels  = 0
    processed     = 0

    with torch.no_grad():
        for images, img_labels, masks_gt in loader:
            images     = images.to(DEVICE)
            img_labels = img_labels.to(DEVICE)

            logits, cam_cls, feats, _ = model(images)
            B, _, H, W = feats.shape

            for b in range(B):
                feat_b  = feats[b]
                cam_b   = cam_cls[b]
                label_b = img_labels[b]
                mask_gt = masks_gt[b].numpy()    # [H, W] ground truth

                try:
                    cam_proto, _ = pot_forward_single(feat_b, cam_b, label_b, H, W)
                    # Upsample CAM al tamaño de la máscara GT
                    cam_up = F.interpolate(
                        cam_proto.unsqueeze(0), size=mask_gt.shape,
                        mode='bilinear', align_corners=False
                    ).squeeze(0)
                    seg_pred = cam_to_segmentation(cam_up).cpu().numpy()  # [H, W]
                    # Convertir índices: cam usa 0=bg,1..C=fg → gt usa 0..C-1 y 255
                    seg_fg = seg_pred - 1   # ahora 0..C-1, bg=-1
                except Exception:
                    seg_fg = np.full(mask_gt.shape, -1, dtype=np.int64)

                # Calcular métricas (ignorar pixels 255)
                valid = (mask_gt != 255)
                pred_valid = seg_fg[valid]
                gt_valid   = mask_gt[valid]

                # Pixel accuracy
                total_correct += (pred_valid == gt_valid).sum()
                total_pixels  += valid.sum()

                # IoU y Dice por clase
                for c in range(NUM_CLASSES):
                    pred_c = (pred_valid == c)
                    gt_c   = (gt_valid == c)
                    inter  = (pred_c & gt_c).sum()
                    uni    = (pred_c | gt_c).sum()
                    intersection[c] += inter
                    union[c]        += uni
                    dice_num[c]     += 2 * inter
                    dice_den[c]     += pred_c.sum() + gt_c.sum()

                processed += 1

            if n_samples and processed >= n_samples:
                break

    # Calcular métricas finales
    iou_per_class  = np.where(union > 0, intersection / union, np.nan)
    dice_per_class = np.where(dice_den > 0, dice_num / dice_den, np.nan)
    miou       = np.nanmean(iou_per_class)
    mean_dice  = np.nanmean(dice_per_class)
    pixel_acc  = total_correct / max(total_pixels, 1)

    return {
        'mIoU'       : float(miou),
        'Dice'       : float(mean_dice),
        'PixelAcc'   : float(pixel_acc),
        'iou_per_class' : iou_per_class,
        'dice_per_class': dice_per_class,
        'n_samples'  : processed
    }


print('✅ Función de evaluación implementada')
print('   Métricas: mIoU | Dice | Pixel Accuracy (ignorando label=255)')

✅ Función de evaluación implementada
   Métricas: mIoU | Dice | Pixel Accuracy (ignorando label=255)


---
## 🏋️ SECCIÓN 11 — Ejecutar entrenamiento completo

Aquí lanzamos el entrenamiento por NUM_EPOCHS épocas, evaluando al final de cada una.
El mejor modelo (según mIoU de validación) se guarda como checkpoint.

In [ ]:
# Agrega esta celda ANTES de la Sección 11
import itertools

grid = {
    'IMG_SIZE'      : [224, 448],
    'CAM_THRESHOLD' : [0.4, 0.7],
}

results = []

for img_size, threshold in itertools.product(grid['IMG_SIZE'], grid['CAM_THRESHOLD']):
    print(f'\n🔍 Probando: IMG_SIZE={img_size} | CAM_THRESHOLD={threshold}')

    # Actualizar hiperparámetros
    IMG_SIZE      = img_size
    CAM_THRESHOLD = threshold

    # Re-crear dataset y modelo con nuevos params
    train_dataset = CityscapesWeakDataset(DATA_ROOT, split='train', transform=transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]))
    train_sub    = Subset(train_dataset, range(500))
    train_loader = DataLoader(train_sub, batch_size=2 if img_size==448 else 4,
                              shuffle=True, num_workers=2, drop_last=True)

    model     = POTEncoder(NUM_CLASSES, FEAT_DIM).to(DEVICE)
    enc_p     = list(model.layer0.parameters()) + list(model.layer1.parameters()) + \
                list(model.layer2.parameters()) + list(model.layer3.parameters()) + \
                list(model.layer4.parameters())
    ext_p     = list(model.hier_proj.parameters()) + list(model.classifier.parameters())
    optimizer = torch.optim.SGD([{'params': enc_p, 'lr': LR_ENCODER},
                                 {'params': ext_p, 'lr': LR_EXTRA}],
                                momentum=0.9, weight_decay=1e-4)

    # Entrenar 3 épocas rápidas
    global_step = 0
    total_steps = 3 * len(train_loader)
    for epoch in range(3):
        train_one_epoch(epoch)

    # Evaluar
    metrics = evaluate(val_loader, n_samples=100)
    results.append({
        'IMG_SIZE': img_size, 'CAM_THRESHOLD': threshold,
        'mIoU': metrics['mIoU'], 'Dice': metrics['Dice']
    })
    print(f'   → mIoU={metrics["mIoU"]:.4f} | Dice={metrics["Dice"]:.4f}')

# Tabla de resultados
print('\n📋 RESULTADOS GRID SEARCH')
print(f'{"IMG_SIZE":<12} {"Threshold":<12} {"mIoU":>8} {"Dice":>8}')
print('-' * 42)
for r in sorted(results, key=lambda x: x['mIoU'], reverse=True):
    print(f'{r["IMG_SIZE"]:<12} {r["CAM_THRESHOLD"]:<12} {r["mIoU"]*100:>7.2f}% {r["Dice"]*100:>7.2f}%')

best = max(results, key=lambda x: x['mIoU'])
print(f'\n🏆 Mejor combinación: IMG_SIZE={best["IMG_SIZE"]} | Threshold={best["CAM_THRESHOLD"]}')


🔍 Probando: IMG_SIZE=224 | CAM_THRESHOLD=0.4
   [train] 2975 muestras cargadas
   Época 1 [0/125] L_cls=0.7344 L_otcon=0.0650 L_total=0.9295 LR=0.10000
   Época 1 [50/125] L_cls=0.3557 L_otcon=0.0343 L_total=0.4585 LR=0.08792
   Época 1 [100/125] L_cls=0.4616 L_otcon=0.0612 L_total=0.6453 LR=0.07564
   Época 2 [0/125] L_cls=0.4333 L_otcon=0.0472 L_total=0.5749 LR=0.06943
   Época 2 [50/125] L_cls=0.4919 L_otcon=0.0457 L_total=0.6290 LR=0.05679
   Época 2 [100/125] L_cls=0.4413 L_otcon=0.0589 L_total=0.6181 LR=0.04384
   Época 3 [0/125] L_cls=0.3506 L_otcon=0.0541 L_total=0.5130 LR=0.03720
   Época 3 [50/125] L_cls=0.3697 L_otcon=0.0655 L_total=0.5663 LR=0.02349
   Época 3 [100/125] L_cls=0.3417 L_otcon=0.0476 L_total=0.4845 LR=0.00874
   → mIoU=0.0202 | Dice=0.0386

🔍 Probando: IMG_SIZE=224 | CAM_THRESHOLD=0.7
   [train] 2975 muestras cargadas
   Época 1 [0/125] L_cls=0.7307 L_otcon=0.0610 L_total=0.9137 LR=0.10000
   Época 1 [50/125] L_cls=0.4263 L_otcon=0.0440 L_total=0.5584 LR=0.08

In [ ]:
# ── Diagnóstico rápido del pipeline POT ─────────────────────────────────────
model.eval()
images, img_labels, masks_gt = next(iter(val_loader))
images     = images.to(DEVICE)
img_labels = img_labels.to(DEVICE)

with torch.no_grad():
    logits, cam_cls, feats, _ = model(images)
    feat_b  = feats[0]
    cam_b   = cam_cls[0]
    label_b = img_labels[0]
    H, W    = feat_b.shape[1], feat_b.shape[2]

    print(f'feat_b shape : {feat_b.shape}')
    print(f'cam_b shape  : {cam_b.shape}')
    print(f'label_b      : {label_b}')
    print(f'Clases presentes: {label_b.nonzero().squeeze()}')

    try:
        cam_proto, loss_otcon = pot_forward_single(feat_b, cam_b, label_b, H, W)
        print(f'✅ POT OK — cam_proto: {cam_proto.shape} | loss_otcon: {loss_otcon:.4f}')
    except Exception as e:
        print(f'❌ POT falló: {e}')
        import traceback
        traceback.print_exc()

---
## 📈 SECCIÓN 12 — Curvas de entrenamiento y métricas por clase

Visualizamos:
1. Evolución de las losses durante el entrenamiento (L_cls, L_otcon, L_total)
2. Evolución de mIoU, Dice y Pixel Accuracy en validación
3. IoU por clase en el conjunto de validación

In [ ]:
# Pega esto AL INICIO de la Sección 12, antes de cualquier otra línea
epochs_range = range(1, len(history['val_miou']) + 1)

# Verificar que todos tienen la misma longitud
n = len(history['val_miou'])
for key in history:
    history[key] = history[key][:n]

print(f'Épocas en historial: {n}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('POT — Cityscapes: Curvas de Entrenamiento', fontsize=15, fontweight='bold')
n_epochs_actual = len(history['val_miou'])
epochs_range    = range(1, n_epochs_actual + 1)

# ── Loss total ──────────────────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(epochs_range, history['train_loss_cls'],   'b-o', label='L_cls')
ax.plot(epochs_range, history['train_loss_otcon'],  'r-s', label='L_otcon')
ax.plot(epochs_range, history['train_loss_total'],  'g-^', label='L_total', linewidth=2)
ax.set_title('Losses de Entrenamiento'); ax.set_xlabel('Época'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

# ── mIoU ────────────────────────────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(epochs_range, history['val_miou'], 'purple', marker='o', linewidth=2)
ax.axhline(y=max(history['val_miou']) if history['val_miou'] else 0,
           color='purple', linestyle='--', alpha=0.5,
           label=f'Best mIoU={max(history["val_miou"]):.4f}' if history['val_miou'] else '')
ax.set_title('mIoU en Validación'); ax.set_xlabel('Época'); ax.set_ylabel('mIoU')
ax.legend(); ax.grid(True, alpha=0.3)

# ── Dice y PixelAcc ──────────────────────────────────────────────────────────
ax = axes[1, 0]
ax.plot(epochs_range, history['val_dice'],      'orange', marker='s', label='Dice')
ax.plot(epochs_range, history['val_pixel_acc'], 'teal',   marker='^', label='Pixel Acc')
ax.set_title('Dice y Pixel Accuracy'); ax.set_xlabel('Época'); ax.set_ylabel('Score')
ax.legend(); ax.grid(True, alpha=0.3)

# ── IoU por clase ────────────────────────────────────────────────────────────
ax = axes[1, 1]
final_metrics = evaluate(val_loader, n_samples=200)
iou_classes   = final_metrics['iou_per_class']
valid_mask    = ~np.isnan(iou_classes)
ax.barh([CITYSCAPES_CLASSES[i] for i in range(NUM_CLASSES) if valid_mask[i]],
        iou_classes[valid_mask],
        color=[np.array(CITYSCAPES_COLORS[i])/255 for i in range(NUM_CLASSES) if valid_mask[i]])
ax.axvline(x=np.nanmean(iou_classes), color='red', linestyle='--',
           label=f'mIoU={np.nanmean(iou_classes):.3f}')
ax.set_title('IoU por Clase (Validación)'); ax.set_xlabel('IoU')
ax.legend(); ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('/content/pot_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráficas guardadas en /content/pot_training_curves.png')

---
## 🖼️ SECCIÓN 13 — Visualizaciones cualitativas

Visualizamos el pipeline completo para algunas imágenes del conjunto de validación:
- Imagen original
- CAM del clasificador (baseline, single-prototype)
- CAM de POT (multi-cluster, nuestro método)
- Máscara predicha
- Ground truth

Esto permite analizar cualitativamente dónde mejora POT respecto al baseline.

In [ ]:
def colorize_mask(mask, ignore_label=255):
    """Convierte máscara de índices a imagen RGB con colores de Cityscapes."""
    H, W   = mask.shape
    color  = np.zeros((H, W, 3), dtype=np.uint8)
    for idx, col in enumerate(CITYSCAPES_COLORS):
        color[mask == idx] = col
    color[mask == ignore_label] = [0, 0, 0]
    return color


def denormalize(tensor):
    """Desnormaliza imagen de tensor a numpy RGB."""
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.cpu().numpy().transpose(1, 2, 0)
    img  = img * std + mean
    return np.clip(img, 0, 1)


def visualize_sample(image_t, img_label, mask_gt, title=''):
    """Visualiza imagen, CAM baseline, CAM POT, predicción y GT."""
    model.eval()
    with torch.no_grad():
        inp    = image_t.unsqueeze(0).to(DEVICE)
        logits, cam_cls, feats, _ = model(inp)
        feat_b = feats[0]
        cam_b  = cam_cls[0]

        H, W = feat_b.shape[1], feat_b.shape[2]

        # CAM del clasificador (baseline)
        cam_cls_up = F.interpolate(cam_b.unsqueeze(0), size=mask_gt.shape,
                                   mode='bilinear', align_corners=False).squeeze(0)
        cam_cls_vis = cam_cls_up[1:].max(0)[0].cpu().numpy()  # max sobre clases fg

        # CAM de POT
        cam_proto, _ = pot_forward_single(feat_b, cam_b, img_label.to(DEVICE), H, W)
        cam_proto_up = F.interpolate(cam_proto.unsqueeze(0), size=mask_gt.shape,
                                     mode='bilinear', align_corners=False).squeeze(0)
        cam_pot_vis  = cam_proto_up[1:].max(0)[0].cpu().numpy()

        # Máscara predicha
        seg_pred = cam_to_segmentation(cam_proto_up).cpu().numpy() - 1
        seg_pred = np.clip(seg_pred, 0, NUM_CLASSES - 1).astype(np.uint8)

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(title, fontsize=12)

    axes[0].imshow(denormalize(image_t.cpu()))
    axes[0].set_title('Imagen Original'); axes[0].axis('off')

    axes[1].imshow(cam_cls_vis, cmap='jet'); axes[1].set_title('CAM Clasificador (baseline)')
    axes[1].axis('off')

    axes[2].imshow(cam_pot_vis, cmap='jet'); axes[2].set_title('CAM POT (nuestro método)')
    axes[2].axis('off')

    axes[3].imshow(colorize_mask(seg_pred))
    axes[3].set_title('Predicción POT'); axes[3].axis('off')

    axes[4].imshow(colorize_mask(mask_gt.numpy().astype(np.uint8)))
    axes[4].set_title('Ground Truth'); axes[4].axis('off')

    plt.tight_layout()
    plt.show()


# ── Visualizar 5 muestras aleatorias de validación ──────────────────────────
print('🎨 Visualizando muestras de validación...')
indices = random.sample(range(len(val_dataset)), min(5, len(val_dataset)))

for i, idx in enumerate(indices):
    image, img_label, mask_gt = val_dataset[idx]
    try:
        visualize_sample(image, img_label, mask_gt, title=f'Muestra {i+1} (val idx={idx})')
    except Exception as e:
        print(f'   Error en muestra {idx}: {e}')

---
## 🔬 SECCIÓN 14 — Ablation Study

Reproducimos el ablation study del paper (Tabla 3 y 4) comparando:
- **Baseline** (I): single-cluster, sin OT, sin consistency loss
- **Condición II**: multi-cluster sin OT
- **Condición III**: multi-cluster + OT vanilla (distribución uniforme)
- **Condición IV (POT completo)**: multi-cluster + Similarity-aware OT + L_otcon

Esto permite verificar la contribución de cada componente en Cityscapes.

In [ ]:
def evaluate_ablation(use_multi_cluster=True, use_sot=True, use_otcon=True, n_samples=100):
    """
    Evalúa distintas configuraciones para el ablation study.

    Args:
        use_multi_cluster: True = K clusters, False = 1 cluster (baseline)
        use_sot          : True = similarity-aware OT, False = OT uniforme
        use_otcon        : True = OT consistency loss activa
    """
    model.eval()
    intersection = np.zeros(NUM_CLASSES)
    union        = np.zeros(NUM_CLASSES)
    processed    = 0
    K_use        = K_CLUSTERS if use_multi_cluster else 1

    with torch.no_grad():
        for images, img_labels, masks_gt in val_loader:
            images     = images.to(DEVICE)
            img_labels = img_labels.to(DEVICE)
            logits, cam_cls, feats, _ = model(images)
            B, _, H, W = feats.shape

            for b in range(B):
                feat_b  = feats[b]
                cam_b   = cam_cls[b]
                label_b = img_labels[b]
                mask_gt = masks_gt[b].numpy()

                try:
                    feats_flat = feat_b.reshape(feat_b.shape[0], -1)
                    cam_flat   = cam_b.reshape(cam_b.shape[0], -1)

                    cluster_protos, cls_protos, _, present = feature_partitioning(
                        feats_flat, cam_flat, label_b, K=K_use
                    )

                    if use_multi_cluster and len(present) > 0:
                        if use_sot:
                            T_star, assignment, _ = feature_allocation(
                                feats_flat, cluster_protos, cls_protos, present
                            )
                        else:
                            # OT uniforme (sin similarity-aware)
                            N = len(present) * K_use
                            x_unif = torch.full((N,), 1.0/N, device=DEVICE)
                            proto_list = [cluster_protos[c] for c in present]
                            all_p = torch.cat(proto_list, dim=0)
                            S = cosine_similarity_matrix(feats_flat.T.float(), all_p)
                            T_star  = sinkhorn_ot(S, x_unif)
                            assignment = T_star.argmax(dim=1)

                        cam_proto, _ = feature_activation(
                            feats_flat, cluster_protos, T_star, assignment, present, H, W
                        )
                    else:
                        # Single-cluster: usar CAM del clasificador directamente
                        cam_proto = cam_b

                    cam_up   = F.interpolate(cam_proto.unsqueeze(0), size=mask_gt.shape,
                                            mode='bilinear', align_corners=False).squeeze(0)
                    seg_pred = (cam_to_segmentation(cam_up).cpu().numpy() - 1)
                    seg_pred = np.clip(seg_pred, 0, NUM_CLASSES - 1)
                except Exception:
                    seg_pred = np.zeros(mask_gt.shape, dtype=np.int64)

                valid = (mask_gt != 255)
                for c in range(NUM_CLASSES):
                    pc = (seg_pred[valid] == c)
                    gc = (mask_gt[valid] == c)
                    intersection[c] += (pc & gc).sum()
                    union[c]        += (pc | gc).sum()

                processed += 1
            if processed >= n_samples:
                break

    iou = np.where(union > 0, intersection / union, np.nan)
    return float(np.nanmean(iou))


# ── Ejecutar ablation ────────────────────────────────────────────────────────
print('🔬 Ejecutando Ablation Study (esto puede tomar varios minutos)...')
print('='*55)

ablation_results = [
    ('I  — Baseline (single-cluster)', False, False, False),
    ('II — Multi-cluster (sin OT)',    True,  False, False),
    ('III— Multi-cluster + OT vanilla',True,  False, True),
    ('IV — POT completo (SOT+Lotcon)', True,  True,  True),
]

ablation_miou = {}
for name, mc, sot, oc in ablation_results:
    print(f'   Evaluando: {name}...')
    miou = evaluate_ablation(mc, sot, oc, n_samples=80)
    ablation_miou[name] = miou
    print(f'   → mIoU = {miou:.4f}')

# ── Tabla resumen ────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('📋 RESUMEN ABLATION STUDY — Cityscapes')
print(f'{"Configuración":<40} {"mIoU":>8}')
print('-'*50)
for name, miou in ablation_miou.items():
    print(f'{name:<40} {miou*100:>7.2f}%')

# ── Gráfico del ablation ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
names  = [k.split('—')[1].strip() for k in ablation_miou.keys()]
values = [v * 100 for v in ablation_miou.values()]
bars   = ax.bar(names, values, color=['#95a5a6','#3498db','#e67e22','#2ecc71'], width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f}%', ha='center', fontweight='bold')
ax.set_ylabel('mIoU (%)'); ax.set_title('Ablation Study — POT en Cityscapes')
ax.set_ylim(0, max(values) * 1.15)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('/content/pot_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📋 SECCIÓN 15 — Reporte final y análisis comparativo

Generamos el reporte final con todas las métricas, análisis por clase,
y discusión de fortalezas/debilidades del método en Cityscapes.

In [ ]:
# ── Evaluación final completa ────────────────────────────────────────────────
print('📊 Evaluación final en conjunto de validación...')
final_metrics = evaluate(val_loader, n_samples=500)

print('\n' + '='*60)
print('📋 REPORTE FINAL — POT en Cityscapes')
print('='*60)
print(f'  mIoU       : {final_metrics["mIoU"]*100:.2f}%')
print(f'  Dice       : {final_metrics["Dice"]*100:.2f}%')
print(f'  Pixel Acc  : {final_metrics["PixelAcc"]*100:.2f}%')
print(f'  Muestras   : {final_metrics["n_samples"]}')
print('\n  IoU por clase:')
iou_cls  = final_metrics['iou_per_class']
dice_cls = final_metrics['dice_per_class']
print(f'  {"Clase":<20} {"IoU":>8} {"Dice":>8}')
print('  ' + '-'*38)
for i, cls_name in enumerate(CITYSCAPES_CLASSES):
    iou_v  = f'{iou_cls[i]*100:.2f}%'  if not np.isnan(iou_cls[i])  else 'N/A'
    dice_v = f'{dice_cls[i]*100:.2f}%' if not np.isnan(dice_cls[i]) else 'N/A'
    print(f'  {cls_name:<20} {iou_v:>8} {dice_v:>8}')

print('\n  Clases con mejor desempeño (Top 5):')
sorted_classes = np.argsort(np.where(np.isnan(iou_cls), -1, iou_cls))[::-1]
for i in sorted_classes[:5]:
    if not np.isnan(iou_cls[i]):
        print(f'    {CITYSCAPES_CLASSES[i]:<20}: {iou_cls[i]*100:.2f}%')

print('\n  Clases con peor desempeño (Bottom 5):')
for i in sorted_classes[-5:][::-1]:
    if not np.isnan(iou_cls[i]):
        print(f'    {CITYSCAPES_CLASSES[i]:<20}: {iou_cls[i]*100:.2f}%')

# ── Guardar reporte ──────────────────────────────────────────────────────────
report = {
    'dataset'   : 'Cityscapes',
    'method'    : 'POT (Prototypical Optimal Transport)',
    'mIoU'      : final_metrics['mIoU'],
    'Dice'      : final_metrics['Dice'],
    'PixelAcc'  : final_metrics['PixelAcc'],
    'iou_per_class' : {CITYSCAPES_CLASSES[i]: float(iou_cls[i])
                       for i in range(NUM_CLASSES) if not np.isnan(iou_cls[i])},
    'ablation'  : ablation_miou,
    'hyperparams': {
        'K_clusters': K_CLUSTERS, 'cam_threshold': CAM_THRESHOLD,
        'theta': THETA, 'lr_encoder': LR_ENCODER, 'epochs': NUM_EPOCHS
    }
}

with open('/content/pot_cityscapes_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print('\n✅ Reporte guardado en /content/pot_cityscapes_report.json')
print('✅ Gráficas: /content/pot_training_curves.png y /content/pot_ablation.png')